# 02. Supervised fine-tuning on Banking77

Supervised fine-tuning is the simplest kind of training: you have examples of the right
answer, and you teach the model to produce them. No rewards, no sampling during training, no
reinforcement learning. Show it the input, show it the output, and the model learns

This notebook does that end to end on **Banking77** which contains 77 customer-service intents that a bank
support message has to be sorted into. It takes about **20 minutes**, most of which is the
model starting up.

The shape of it:

```
1. build a training client       
2. turn chat rows into datums     
3. loop: forward_backward, optim_step
4. publish the adapter, sample from it
5. score it against the recorded baseline
```

Step 5 is the one that matters. It is easy to write a training loop whose loss goes down and
whose accuracy does not move, so this notebook measures the model **before** training and
**after**, through the same code path, and prints both.

You should have been through [01](01_hello_crusoe_mill.ipynb) first: sessions, sampling, and
chat templates are assumed here.

## Setup

Same as 01: install the bundled SDK wheel into this kernel, then authenticate.

In [1]:
try:
    import pip
except ImportError:
    import ensurepip
    ensurepip.bootstrap()

%pip install --quiet --upgrade -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("CRUSOE_API_KEY"):
    os.environ["CRUSOE_API_KEY"] = getpass("Crusoe API key: ")

os.environ.setdefault("CRUSOE_BASE_URL", "https://api.intelligence.crusoecloud.com")

In [3]:
MODEL = "Qwen/Qwen3.5-9B"

LORA_RANK = 16   # must be <= the model's maximum rank on the service
LEARNING_RATE = 1e-4  
BATCH_ROWS = 32        
EPOCHS = 1

EVAL_ROWS = 231         
ANSWER_MAX_TOKENS = 16 

# What this model scores on these 231 test rows BEFORE any training, measured separately, recorded number
BASE_ACCURACY = 0.71

## The data

The next cell builds three files under `data/banking77/` from the public Banking77 dataset,
pinned by commit and checked by md5. Each row is a chat conversation:
a system prompt listing all 77 intents, the customer's message, and the correct intent as the
assistant's reply.

| file | rows | what it is for |
|---|---|---|
| `train.jsonl` | 1,001 | the model trains on these |
| `validation.jsonl` | 231 | spare, for tuning if you want it |
| `test.jsonl` | 231 | held out, scored before and after training |

All three are stratified so every one of the 77 intents appears in each file. They are subsets
of the full dataset, small enough to keep this notebook quick.

In [4]:
import json
from pathlib import Path

from prepare_data import ensure_banking77

EXAMPLE_DATA = Path("data/banking77")
# To train on your own data, point this at a directory holding train.jsonl and test.jsonl.
DATA = EXAMPLE_DATA

if DATA == EXAMPLE_DATA:
    # Builds the splits from the public upstream dataset, pinned by commit, and checks each
    # file's md5. Already built means this is a no-op.
    ensure_banking77()


def load_rows(name):
    return [json.loads(line) for line in (DATA / name).read_text().splitlines() if line.strip()]


def answer_of(row):
    """The assistant turn: the intent this row is graded against."""
    return next(m["content"] for m in row["messages"] if m["role"] == "assistant").strip()


train_rows = load_rows("train.jsonl")
test_rows = load_rows("test.jsonl")[:EVAL_ROWS]

example = train_rows[0]
for message in example["messages"]:
    body = message["content"]
    print(f"{message['role']:>9}: {body if len(body) < 160 else body[:160] + ' ...'}")
print(f"\n{len(train_rows)} training rows, {len(test_rows)} test rows")

  ok       train.jsonl             1001 rows  c93851b883aeb5e9f912b40da3f04a75
  ok       validation.jsonl         231 rows  9fef123cf093c22969f0577e56ef6fa2
  ok       test.jsonl               231 rows  b06bd32c5d70500f02fe71fb15f6047a
banking77 already built and verified
   system: You are a banking customer service intent classifier. Given a customer message, classify it into exactly one of these intents:

activate_my_card, age_limit, app ...
     user: How can I stop someone from using my card after I lost it yesterday.
assistant: lost_or_stolen_phone

1001 training rows, 231 test rows


## 1. The training client

This is the object that trains. You get it from the same `ServiceClient` as in 01, and the
session has to declare the model as a **training** model plus a sampling row, because we
sample from the adapter later to score it.

```python
training_client = client.create_lora_training_client(base_model=MODEL, rank=16)
```

**It trains a LoRA adapter, not the model.** The base weights stay frozen so what you train is a
small pair of low-rank matrices added alongside them. That is why this is affordable, and why
what you save at the end is a few hundred megabytes rather than the whole model.

**The `ServiceClient` in the next cell is what waits for your GPUs.** Constructing it with a
`ServiceConfig` provisions the session and blocks until it is ready, minutes on a cold start.
`create_lora_training_client` then asks that ready session for a trainer and returns quickly.

In [5]:
from crusoe_mill import (
    SamplingBaseModelConfig,
    ServiceClient,
    ServiceConfig,
    TrainingBaseModelConfig,
)

client = ServiceClient(
    service_config=ServiceConfig(
        training_client_base_models=[TrainingBaseModelConfig(base_model=MODEL)],
        # Declared too, because step 3 and step 5 sample from the adapter to score it. 
        # A session only gets sampler capacity for models it asked for.
        sampling_client_base_models=[SamplingBaseModelConfig(base_model=MODEL)],
    ),
)


training_client = client.create_lora_training_client(
    base_model=MODEL,
    rank=LORA_RANK,
    train_attn=True,
    train_mlp=True,
    train_unembed=False,
    seed=0,
)
print("training client ready")

[2026-09-28 21:46:08 crusoe_mill._progress INFO] tinker session msess_2517d36702fb42cfaa3935e27e9c144b created; provisioning Qwen/Qwen3.5-9B
[2026-09-28 21:46:08 crusoe_mill.service_client INFO] ServiceClient initialized for session msess_2517d36702fb42cfaa3935e27e9c144b
[2026-09-28 21:46:08 crusoe_mill._progress INFO] provisioning — 2s elapsed (session msess_2517d36702fb42cfaa3935e27e9c144b; ^C releases it)
[2026-09-28 21:47:25 crusoe_mill._progress INFO] provisioning — 79s elapsed (session msess_2517d36702fb42cfaa3935e27e9c144b; ^C releases it)
[2026-09-28 21:48:26 crusoe_mill._progress INFO] provisioning — 140s elapsed (session msess_2517d36702fb42cfaa3935e27e9c144b; ^C releases it)
[2026-09-28 21:49:27 crusoe_mill._progress INFO] provisioning — 201s elapsed (session msess_2517d36702fb42cfaa3935e27e9c144b; ^C releases it)
[2026-09-28 21:50:29 crusoe_mill._progress INFO] provisioning — 263s elapsed (session msess_2517d36702fb42cfaa3935e27e9c144b; ^C releases it)
[2026-09-28 21:51:30 

training client ready


## 2. From chat rows to datums

The service trains on `Datum` objects, not on text, so each row has to be turned into token
ids plus an instruction about which positions to learn from.

**Only the answer should carry loss.** The system prompt here is ~430 tokens listing every
intent, and the answer is ~6. If you train on the whole sequence, almost all of the gradient
comes from memorising the prompt you already have, and the thing you actually care about is
lost in the noise.

So we tokenise in two pieces - the prompt (ending with the "assistant speaks now" marker) and
the answer.

Then the shift. At position `i` the model predicts the token at `i + 1`. So:

```
model_input = input_ids[:-1]          # everything except the last token
target_tokens = input_ids[1:]         # what should come next at each position
weights[i] = 1.0 if i + 1 is inside the answer else 0.0
```

**`weights` is the mask**, and getting that `+ 1` wrong is the classic way to waste a training
run: the loss still falls, because the model learns to echo the token it was just handed, and
nothing looks broken. The cell after this one decodes the masked positions back to text so you
can see with your own eyes that they are exactly the intent.

In [6]:
from tinker import types
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)

END_OF_TURN = tokenizer.convert_tokens_to_ids("<|im_end|>")


def encode(messages):
    """(input_ids, prompt_length). Loss belongs on everything from prompt_length onward."""
    prompt_messages = [m for m in messages if m["role"] != "assistant"]
    answer = next(m["content"] for m in messages if m["role"] == "assistant").strip()

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=False,
        enable_thinking=False,
    )
    # The end-of-turn token is part of what we teach: without it the model never stops.
    answer_ids = tokenizer.encode(answer, add_special_tokens=False) + [END_OF_TURN]
    return prompt_ids + answer_ids, len(prompt_ids)


def shift_for_training(input_ids, prompt_length):
    """(model_input, target_tokens, weights). At position i the model predicts input_ids[i+1],
    so loss belongs where that predicted token is inside the answer."""
    return (
        input_ids[:-1],
        input_ids[1:],
        [1.0 if i + 1 >= prompt_length else 0.0 for i in range(len(input_ids) - 1)],
    )


def to_datum(input_ids, prompt_length):
    model_input, target_tokens, weights = shift_for_training(input_ids, prompt_length)
    return types.Datum(
        model_input=types.ModelInput.from_ints(model_input),
        loss_fn_inputs={"target_tokens": target_tokens, "weights": weights},
    )

### Check the mask before you train

Decode exactly the positions that carry loss. They should spell the intent and nothing else.

In [7]:
input_ids, prompt_length = encode(example["messages"])

_, targets, weights = shift_for_training(input_ids, prompt_length)
supervised = [t for t, w in zip(targets, weights) if w > 0]

print("prompt ends with :", repr(tokenizer.decode(input_ids[prompt_length - 8:prompt_length])))
print("we train on      :", repr(tokenizer.decode(supervised)))
print("gold answer      :", repr(answer_of(example)))
print()
print(f"{int(sum(weights))} of {len(weights)} positions carry loss "
      f"({100 * sum(weights) / len(weights):.1f}%)")

assert tokenizer.decode(supervised, skip_special_tokens=True).strip() == answer_of(example), \
    "the mask does not line up with the answer — do not train until this passes"
print("\nmask checks out")

prompt ends with : '\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
we train on      : 'lost_or_stolen_phone<|im_end|>'
gold answer      : 'lost_or_stolen_phone'

6 of 444 positions carry loss (1.4%)

mask checks out


In [8]:
import random

encoded = [encode(row["messages"]) for row in train_rows]
random.Random(0).shuffle(encoded)

steps_per_epoch = -(-len(encoded) // BATCH_ROWS)      
TOTAL_STEPS = steps_per_epoch * EPOCHS
print(f"{len(encoded)} rows / {BATCH_ROWS} per step = {steps_per_epoch} steps/epoch"
      f" -> {TOTAL_STEPS} steps")

1001 rows / 32 per step = 32 steps/epoch -> 32 steps


## 3. The scoring helpers

Training is next, but the pieces that score the result are worth reading first, because the
number they produce is the whole point of the notebook.

**We already measured base model here.** It scores about **71%** on these 231 rows, measured
separately under exactly this protocol, and that figure is in the config cell as
`BASE_ACCURACY`. Measuring it in-notebook would mean publishing an untrained adapter and a
second full pass over the test set, which roughly doubles the sampling for a number that does
not change. The trade is that the baseline is inherited rather than observed so if you
change the model, the test split, or the grader, that constant is no longer yours and you
should re-measure it.

**`publish` is two calls.** `save_weights_for_sampler` writes the adapter where a sampler can
reach it and returns a path; `create_sampling_client` then serves that path. The path outlives
the session, so you can sample from the adapter later without retraining.

In [9]:
def publish(tag):
    """Save the adapter as it stands and return a sampling client for it."""
    path = training_client.save_weights_for_sampler(tag).result().path
    return client.create_sampling_client(model_path=path), path


def evaluate(sampling_client, rows, label):
    """Greedy-sample each prompt, exact-match the intent, return accuracy."""
    params = types.SamplingParams(max_tokens=ANSWER_MAX_TOKENS, temperature=0.0)

    futures = []
    for row in rows:
        prompt_messages = [m for m in row["messages"] if m["role"] != "assistant"]
        prompt_ids = tokenizer.apply_chat_template(
            prompt_messages, add_generation_prompt=True, tokenize=True,
            return_dict=False, enable_thinking=False,
        )
        futures.append(sampling_client.sample(
            prompt=types.ModelInput.from_ints(prompt_ids),
            sampling_params=params,
            num_samples=1,
        ))

    correct, predictions = 0, []
    for row, future in zip(rows, futures):
        text = tokenizer.decode(future.result().sequences[0].tokens,
                                skip_special_tokens=True).strip()
        predictions.append(text)
        correct += int(text == answer_of(row))

    accuracy = correct / len(rows)
    print(f"{label}: {accuracy:.1%} ({correct}/{len(rows)})")
    return accuracy, predictions

## 4. The training loop

This is the whole of it. Two calls per step:

- **`forward_backward(datums, "cross_entropy")`** computes the loss and the gradients. The
  `weights` in each datum are what confine the loss to the answer tokens.
- **`optim_step(AdamParams(...))`** applies them.

Both return futures, so we submit them back to back and only then wait, which keeps the
optimizer from idling while the backward pass finishes. Wait on the optimizer first, as the
code comment explains; notebook 03 does the same.

The loss we print is **per supervised token** (`loss:sum` divided by the number of tokens that
carried loss), not per batch. Batches differ in length, so a per-batch figure is not comparable
between steps.

In [10]:
import math
import time


def cosine_lr(step):
    """Peak at step 0, ~0 at the last step. No warmup."""
    if TOTAL_STEPS <= 1:
        return LEARNING_RATE
    progress = min(1.0, step / (TOTAL_STEPS - 1))
    return LEARNING_RATE * 0.5 * (1.0 + math.cos(math.pi * progress))


history = []

for step in range(TOTAL_STEPS):
    started = time.perf_counter()

    start = (step * BATCH_ROWS) % len(encoded)
    batch = encoded[start:start + BATCH_ROWS]
    if len(batch) < BATCH_ROWS:
        batch += encoded[:BATCH_ROWS - len(batch)]
    datums = [to_datum(ids, prompt_length) for ids, prompt_length in batch]
    learning_rate = cosine_lr(step)

    forward_backward = training_client.forward_backward(datums, "cross_entropy")
    optimizer = training_client.optim_step(
        types.AdamParams(learning_rate=learning_rate, beta1=0.9, beta2=0.999, eps=1e-8))
    # Optimizer first: a large batch can be sent in several parts, and the step completes only
    # once all of them have arrived, so waiting on it first guarantees the whole batch is in.
    optimizer.result()
    metrics = dict(forward_backward.result().metrics)

    denominator = ("unmasked_tokens:sum" if "unmasked_tokens:sum" in metrics
                   else "num_tokens:sum")
    loss = metrics["loss:sum"] / max(1.0, metrics[denominator])

    history.append({"step": step, "loss": loss, "lr": learning_rate,
                    "seconds": time.perf_counter() - started})
    print(f"step {step + 1:3d}/{TOTAL_STEPS}  loss {loss:.4f}  "
          f"lr {learning_rate:.2e}  {history[-1]['seconds']:.1f}s")

step   1/32  loss 0.0041  lr 1.00e-04  69.9s
step   2/32  loss 0.0035  lr 9.97e-05  17.3s
step   3/32  loss 0.0029  lr 9.90e-05  1.5s
step   4/32  loss 0.0022  lr 9.77e-05  1.7s
step   5/32  loss 0.0029  lr 9.59e-05  1.3s
step   6/32  loss 0.0010  lr 9.37e-05  1.5s
step   7/32  loss 0.0029  lr 9.10e-05  1.3s
step   8/32  loss 0.0028  lr 8.79e-05  1.2s
step   9/32  loss 0.0012  lr 8.44e-05  1.3s
step  10/32  loss 0.0037  lr 8.06e-05  1.3s
step  11/32  loss 0.0021  lr 7.64e-05  1.2s
step  12/32  loss 0.0015  lr 7.20e-05  1.3s
step  13/32  loss 0.0015  lr 6.74e-05  1.3s
step  14/32  loss 0.0012  lr 6.25e-05  1.3s
step  15/32  loss 0.0021  lr 5.76e-05  1.4s
step  16/32  loss 0.0022  lr 5.25e-05  1.4s
step  17/32  loss 0.0018  lr 4.75e-05  2.0s
step  18/32  loss 0.0016  lr 4.24e-05  1.3s
step  19/32  loss 0.0023  lr 3.75e-05  1.3s
step  20/32  loss 0.0010  lr 3.26e-05  1.2s
step  21/32  loss 0.0012  lr 2.80e-05  1.3s
step  22/32  loss 0.0017  lr 2.36e-05  1.3s
step  23/32  loss 0.0020  lr 1

## 5. Did it work?

Publish the trained adapter and score the same held-out rows, with the same grader.

In [11]:
trained_sampler, adapter_path = publish("banking77-final")
trained_accuracy, predictions = evaluate(trained_sampler, test_rows, "after training")

print()
print(f"{'base (recorded)':18} {BASE_ACCURACY:>7.1%}")
print(f"{'after training':18} {trained_accuracy:>7.1%}")
print(f"{'change':18} {trained_accuracy - BASE_ACCURACY:>+7.1%}")
print(f"\nadapter saved at: {adapter_path}")

after training: 83.1% (192/231)

base (recorded)      71.0%
after training       83.1%
change              +12.1%

adapter saved at: mill://msess_2517d36702fb42cfaa3935e27e9c144b/sampler_weights/banking77-final


### Reading the result

Accuracy on held-out rows is the claim. The training loss is not as it only says the model fit
the rows it saw.

Compared against the recorded 71% baseline: **up** means the loop did its job.

**Nowhere** points at the mask first (re-run the check in step 2), then too few steps, then a
learning rate too low to move a rank-16 adapter.

**Down** usually means the learning rate is too high and the adapter has overwritten something
the base model already did correctly. Halve it and run again.

Then look at what the mistakes actually are. 

In [12]:
wrong = [(answer_of(row), predicted)
         for row, predicted in zip(test_rows, predictions)
         if predicted != answer_of(row)]

print(f"{len(wrong)} wrong out of {len(test_rows)}. The first few:\n")
for gold, predicted in wrong[:8]:
    in_label_set = predicted in {answer_of(r) for r in test_rows}
    note = "a real intent, wrong one" if in_label_set else "NOT a valid intent"
    print(f"  gold {gold:38} predicted {predicted!r:40} {note}")

39 wrong out of 231. The first few:

  gold balance_not_updated_after_bank_transfer predicted 'transfer_timing'                        a real intent, wrong one
  gold fiat_currency_support                  predicted 'exchange_via_app'                       a real intent, wrong one
  gold balance_not_updated_after_cheque_or_cash_deposit predicted 'top_up_by_cash_or_cheque'               a real intent, wrong one
  gold card_payment_not_recognised            predicted 'compromised_card'                       a real intent, wrong one
  gold topping_up_by_card                     predicted 'top_up_by_card_charge'                  a real intent, wrong one
  gold topping_up_by_card                     predicted 'beneficiary_not_allowed'                a real intent, wrong one
  gold country_support                        predicted 'card_acceptance'                        a real intent, wrong one
  gold fiat_currency_support                  predicted 'exchange_via_app'                       a

## Close the session

The session holds GPUs until you release it. The adapter path printed above outlives the
session, so you can sample from it later without retraining.

In [13]:
client.close()
print("session closed")

session closed


## What to change next

The loop above is generic so nothing in it knows about banking. To use your own data, set
`DATA` in the data cell to a directory holding `train.jsonl` and `test.jsonl` in the same shape:

```json
{"messages": [{"role": "system", "content": "..."},
              {"role": "user", "content": "..."},
              {"role": "assistant", "content": "the answer to learn"}]}
```

Worth trying, roughly in order of effect: more training rows, a couple of epochs instead of
one, then `LORA_RANK` if the task is harder than classification.